In [1]:
%%capture
!wget https://raw.githubusercontent.com/karpathy/makemore/refs/heads/master/names.txt

In [1]:
import torch

In [3]:
names = open('data/names.txt').read().splitlines()

In [4]:
names[:10]

['emma',
 'olivia',
 'ava',
 'isabella',
 'sophia',
 'charlotte',
 'mia',
 'amelia',
 'harper',
 'evelyn']

In [5]:
len(names)

32033

In [6]:
t = {}

for name in names:
    name = ['<s>', '<s>'] + list(name) + ['</s>']  # two start tokens

    for ch1, ch2, ch3 in zip(name, name[1:], name[2:]):
        trigram = (ch1, ch2, ch3)
        t[trigram] = t.get(trigram, 0) + 1


In [7]:
len(t)

6063

In [8]:
sorted(t.items(), key=lambda x: -x[1])[:20]

[(('<s>', '<s>', 'a'), 4410),
 (('<s>', '<s>', 'k'), 2963),
 (('<s>', '<s>', 'm'), 2538),
 (('<s>', '<s>', 'j'), 2422),
 (('<s>', '<s>', 's'), 2055),
 (('a', 'h', '</s>'), 1714),
 (('<s>', '<s>', 'd'), 1690),
 (('n', 'a', '</s>'), 1673),
 (('<s>', '<s>', 'r'), 1639),
 (('<s>', '<s>', 'l'), 1572),
 (('<s>', '<s>', 'c'), 1542),
 (('<s>', '<s>', 'e'), 1531),
 (('a', 'n', '</s>'), 1509),
 (('o', 'n', '</s>'), 1503),
 (('<s>', 'm', 'a'), 1453),
 (('<s>', '<s>', 't'), 1308),
 (('<s>', '<s>', 'b'), 1306),
 (('<s>', 'j', 'a'), 1255),
 (('<s>', 'k', 'a'), 1254),
 (('e', 'n', '</s>'), 1217)]

In [9]:
probs = {}

for (ch1, ch2, ch3), count in t.items():
    if (ch1, ch2) not in probs:
        probs[(ch1, ch2)] = {}
    probs[(ch1, ch2)][ch3] = probs[(ch1, ch2)].get(ch3, 0) + count


In [10]:
for key in probs:
    total = sum(probs[key].values())
    for ch3 in probs[key]:
        probs[key][ch3] /= total

In [11]:
import random

def sample_3gram():
    out = ['<s>', '<s>']  # start with two start tokens

    while True:
        ch1, ch2 = out[-2], out[-1]

        if (ch1, ch2) not in probs:
            break

        next_chars = list(probs[(ch1, ch2)].keys())
        next_probs = list(probs[(ch1, ch2)].values())

        ch3 = random.choices(next_chars, weights=next_probs)[0]

        if ch3 == '</s>':
            break
        out.append(ch3)

    return ''.join(out[2:])

In [12]:
for _ in range(20):
    print(sample_3gram())

sabduliah
kaclearrettson
abbianseamanuskie
io
vajilazebechri
lailinani
kamorbea
jaishna
khyah
mir
charinenshamosyn
criya
nuanslena
lassa
deresly
xzarin
adilyonaya
jaira
gyiah
withmone


## With Tensor

In [13]:
chars = sorted(list(set(''.join(names))))
chars = ['<s>', '</s>'] + chars   # include special tokens
stoi = {c:i for i,c in enumerate(chars)}
itos = {i:c for c,i in stoi.items()}

C = len(chars)
C


28

In [14]:
T = torch.zeros((C, C, C), dtype=torch.int32)

for name in names:
    seq = ['<s>', '<s>'] + list(name) + ['</s>']

    for ch1, ch2, ch3 in zip(seq, seq[1:], seq[2:]):
        i = stoi[ch1]
        j = stoi[ch2]
        k = stoi[ch3]
        T[i, j, k] += 1

In [15]:
(T > 0).sum()

tensor(6063)

In [16]:
P = T.float()
P /= P.sum(dim=2, keepdim=True)  # normalize over next-char dimension

In [17]:
import random

def sample_3gram_tensor():
    out = ['<s>', '<s>']  # start with two start tokens

    while True:
        i = stoi[out[-2]]
        j = stoi[out[-1]]

        probs = P[i, j]

        # If row is all zeros, break
        if probs.sum() == 0:
            break

        # Sample next index
        k = torch.multinomial(probs, num_samples=1).item()
        ch = itos[k]

        if ch == '</s>':
            break

        out.append(ch)

    return ''.join(out[2:])


In [18]:
for _ in range(20):
    print(sample_3gram_tensor())

emisosamickenilian
ulia
brajvington
chrindessijaellio
chrie
deo
miyah
jovana
goree
dys
blaiuhasley
asabhaylyn
ber
maaylahmalivyn
dayohnee
camdy
ta
malynne
brajameemakaidiel
ailvee
